# ResGRU-UNet PM25 - verify

Checks that the files published as the Kaggle model [imvetri0/resgru-unet-pm25](https://www.kaggle.com/models/imvetri0/resgru-unet-pm25) (PyTorch, default, version 1) reproduce the five-seed forecast that was submitted to the test set (private 0.8914 with calibration).

- Each of the five networks loads strictly into `ResGRUUNet(20)` and has 7,122,113 parameters.
- `predict.py` equals `preds_resgru_5seed_cal.npy` and `predict.py --no-cal` equals `preds_resgru_5seed.npy`, both from the 5-seed analysis notebook, each with relative difference mean|a - b| / mean|b| < 1e-3 (the study's threshold).
- Both outputs have shape (218, 140, 124, 16) and are finite and non-negative.

Output: `verify.json` with the checks, both relative differences and the maximum absolute differences.

In [ ]:
import os, sys, json, time, subprocess
import numpy as np
import torch

T0 = time.time()
MODEL_ID = 'imvetri0/resgru-unet-pm25/PyTorch/default/1'
SEEDS = [42, 1337, 1, 2, 3]
SHAPE = (218, 140, 124, 16)
TOL = 1e-3
WORK = '/kaggle/working'


def find(name):
    for dp, _, fs in os.walk('/kaggle/input'):
        if name in fs:
            return dp
    raise FileNotFoundError(f'{name} not found under /kaggle/input')


MODEL, REF = find('predict.py'), find('preds_resgru_5seed_cal.npy')
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print('model    :', MODEL)
print('reference:', REF)
print('files    :', sorted(os.listdir(MODEL)))
print('torch', torch.__version__, '| GPU', GPU)

## 1. The five networks

In [ ]:
sys.path.insert(0, MODEL)
import study as S

checks, nets = {}, {}
for s in SEEDS:
    ck = torch.load(os.path.join(MODEL, f'resgru_s{s}_phaseA_best.pt'), map_location='cpu', weights_only=False)
    m = S.ResGRUUNet(S.C.C_IN)
    m.load_state_dict(ck['model'], strict=True)
    n = sum(p.numel() for p in m.parameters())
    nets[s] = dict(params=n, epoch=ck['epoch'], logged_val_score=round(float(ck['metrics']['score']), 5))
    checks[f'seed {s}: loads strictly into ResGRUUNet(20), 7,122,113 parameters'] = n == 7122113
    print(f'seed {s}: {nets[s]}')
del m, ck

## 2. Run predict.py with and without calibration

In [ ]:
def run(*args):
    r = subprocess.run([sys.executable, os.path.join(MODEL, 'predict.py'), *args], capture_output=True, text=True)
    print(r.stdout, r.stderr)
    assert r.returncode == 0, f'predict.py exited with {r.returncode}'
    out = os.path.join(WORK, 'preds_nocal.npy' if args else 'preds_cal.npy')
    os.replace(os.path.join(WORK, 'preds.npy'), out)
    return np.load(out)


cal = run()
nocal = run('--no-cal')

## 3. Compare with the submitted forecasts

In [ ]:
def diff(a, b):
    d = np.abs(a.astype(np.float64) - b)
    return dict(rel_diff=float(d.mean() / np.abs(b).mean(dtype=np.float64)), max_abs_diff=float(d.max()))


res = dict(model=MODEL_ID, reference='imvetri0/code4cleanair-backbone-study-5-seed-analysis', threshold=TOL, networks=nets)
for tag, a, ref, cmd in (('calibrated', cal, 'preds_resgru_5seed_cal.npy', 'predict.py'),
                         ('uncalibrated', nocal, 'preds_resgru_5seed.npy', 'predict.py --no-cal')):
    b = np.load(os.path.join(REF, ref))
    res[tag] = dict(command=cmd, reference=ref, **diff(a, b), min=float(a.min()), max=float(a.max()), mean=float(a.mean()))
    checks[f'{cmd} output {SHAPE}, finite, non-negative'] = a.shape == SHAPE and bool(np.isfinite(a).all()) and float(a.min()) >= 0
    checks[f'{ref} has shape {SHAPE}'] = b.shape == SHAPE
    checks[f'{cmd} output equals {ref} (rel < {TOL:g})'] = res[tag]['rel_diff'] < TOL
    print(tag, res[tag])
    del b
res.update(checks=checks, all_checks_pass=all(checks.values()), gpu=GPU, torch=torch.__version__,
           minutes=round((time.time() - T0) / 60, 1))
json.dump(res, open(os.path.join(WORK, 'verify.json'), 'w'), indent=1)
for k, v in checks.items():
    print('PASS' if v else 'FAIL', k)
assert res['all_checks_pass'], 'a check failed'